In [1]:
import pandas as pd
from scipy.sparse import coo_matrix

df = pd.read_csv("ratings_Amazon_Instant_Video.csv")

df.columns = ['user', 'item', 'rating', 'timestamp']

# encoding
user_codes = df['user'].astype("category").cat.codes
item_codes = df['item'].astype("category").cat.codes

# mapping balik
user_map = dict(enumerate(df['user'].astype("category").cat.categories))
item_map = dict(enumerate(df['item'].astype("category").cat.categories))

user_inv_map = {v:k for k,v in user_map.items()}
item_inv_map = {v:k for k,v in item_map.items()}

# sparse matrix
matrix = coo_matrix((df['rating'], (user_codes, item_codes)))

In [2]:
df

,user,item,rating,timestamp
0,AGZ8SM1BGK3CK,B000GFDAUG,5.0,1.198195e+09
1,A2VHZ21245KBT7,B000GIOPK2,4.0,1.215389e+09
2,ACX8YW2D5EGP6,B000GIOPK2,4.0,1.185840e+09
3,A9RNMO9MUSMTJ,B000GIOPK2,2.0,1.281053e+09
4,A3STFVPM8NHJ7B,B000GIOPK2,5.0,1.203898e+09
...,...,...,...,...
180148,A2MB3JRUGZR5DN,B005KP7EK4,4.0,1.388534e+09
180149,A1DOLH1E2P8TWY,B005KP7EK4,4.0,1.394582e+09
180150,A36ZGLMMAVR9DY,B005KP7EK4,2.0,1.349568e+09
180151,A7LC8OXP6T9IQ,B005KP7EK4,4.0,1.388448e+09


In [3]:
from sklearn.neighbors import NearestNeighbors

item_matrix = matrix.T  # item-based

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(item_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [4]:
import numpy as np

def recommend_items(user_id, top_n=5):
    if user_id not in user_inv_map:
        return []

    user_idx = user_inv_map[user_id]

    # ambil semua item yang pernah dirating user
    user_vector = matrix.getrow(user_idx)
    rated_items = user_vector.nonzero()[1]

    scores = {}

    for item_idx in rated_items:
        distances, indices = model.kneighbors(
            item_matrix.getrow(item_idx),
            n_neighbors=10
        )

        for i, dist in zip(indices.flatten(), distances.flatten()):
            if i not in rated_items:  # skip yang sudah dirating
                scores[i] = scores.get(i, 0) + (1 - dist)

    # ranking
    ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # ambil top N
    recommendations = [item_map[i] for i, _ in ranked_items[:top_n]]

    return recommendations

In [5]:
user_id = "A2VHZ21245KBT7"
print(recommend_items(user_id, top_n=5))

['B000LVKG2A', 'B000OGTRC2', 'B000W9H2M8', 'B002KGXAMC', 'B003KYZI5U']


In [6]:
import pandas as pd

def get_user_summary(user_id, top_n=5):
    if user_id not in user_inv_map:
        return "User ID tidak ditemukan.", []

    user_idx = user_inv_map[user_id]

    # --- 1. AMBIL DATA RIWAYAT (YANG SUDAH DIBELI) ---
    user_vector = matrix.getrow(user_idx)
    # Ambil index item yang tidak nol (pernah dirating)
    _, purchased_indices = user_vector.nonzero()
    # Ambil nilai ratingnya
    actual_ratings = user_vector.data

    purchased_list = []
    for idx, rating in zip(purchased_indices, actual_ratings):
        purchased_list.append({
            'item_id': item_map[idx],
            'rating': rating
        })

    # Urutkan riwayat berdasarkan rating tertinggi agar kita tahu selera user
    purchased_list = sorted(purchased_list, key=lambda x: x['rating'], reverse=True)

    # --- 2. PROSES REKOMENDASI (ITEM-BASED) ---
    scores = {}
    # Kita hanya mencari kemiripan berdasarkan item yang diberi rating tinggi oleh user (>3)
    # Agar rekomendasi tidak "tercemar" oleh barang yang pernah dibeli tapi dibenci user
    high_rated_indices = [idx for idx, rat in zip(purchased_indices, actual_ratings) if rat >= 3]

    for item_idx in high_rated_indices:
        distances, indices = model.kneighbors(
            item_matrix.getrow(item_idx),
            n_neighbors=10
        )

        for i, dist in zip(indices.flatten(), distances.flatten()):
            # Filter: Jika belum pernah dibeli
            if i not in purchased_indices:
                scores[i] = scores.get(i, 0) + (1 - dist)

    ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    recommendations = []
    for i, score in ranked_items[:top_n]:
        recommendations.append({
            'item_id': item_map[i],
            'confidence_score': round(score, 4)
        })

    return purchased_list, recommendations

# --- EKSEKUSI DAN TAMPILKAN ---
user_id = "A2VHZ21245KBT7"
history, recs = get_user_summary(user_id)

print(f"=== ANALISIS USER: {user_id} ===")
print("\n[1] BARANG YANG SUDAH DIBELI (Riwayat):")
for item in history:
    print(f"- Item: {item['item_id']} | Rating: {item['rating']}")

print("\n[2] BARANG DISARANKAN (Rekomendasi Baru):")
if not recs:
    print("- Tidak ada rekomendasi yang cukup kuat.")
for i, item in enumerate(recs, 1):
    print(f"{i}. Item: {item['item_id']} | Skor Kecocokan: {item['confidence_score']}")

=== ANALISIS USER: A2VHZ21245KBT7 ===

[1] BARANG YANG SUDAH DIBELI (Riwayat):
- Item: B0013FJOG2 | Rating: 5.0
- Item: B000UUEEIE | Rating: 5.0
- Item: B000GIOPK2 | Rating: 4.0
- Item: B004GEYL9M | Rating: 3.0

[2] BARANG DISARANKAN (Rekomendasi Baru):
1. Item: B000LVKG2A | Skor Kecocokan: 0.282
2. Item: B000OGTRC2 | Skor Kecocokan: 0.1596
3. Item: B000W9H2M8 | Skor Kecocokan: 0.1286
4. Item: B002KGXAMC | Skor Kecocokan: 0.0945
5. Item: B003KYZI5U | Skor Kecocokan: 0.0673
